# Multi step workflow with Language Chain Expression Language

Create an AI Business Advisor that:

* Accepts an industry as input.
* Generates a business idea.
* Analyzes the strengths and weaknesses.
* Formats the results as a final report.

Use LCEL to chain prompts LLMs and output parsers.

In [ ]:
from dotenv import load_dotenv

loaded = load_dotenv("../.env")
print("Loaded:", loaded)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from pydantic import BaseModel, Field
import os

In [ ]:
# intantiate the LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openai.vocareum.com/v1"
)

## Create a chain
This runnable should parse the provided input and also log it. So parsing and logging can run in parallel. Hence use RunnableParallel.

RunnableParallel, expects a map of steps. Below we have output and log but these names can be also different.

In [ ]:
logs = []

In [ ]:
parser = StrOutputParser()

In [ ]:
parse_and_log_output_chain = RunnableParallel(
    output=parser,
    log=RunnableLambda(lambda x: logs.append(x))
)

## Idea Generation

Given industry LLM should generate some idea.

In [ ]:
idea_prompt = PromptTemplate(
    template=("""You are an experienced business consultant.
    
Generate one innovative and practical business idea in the {industry} industry.

For the idea, provide:
- Business Name
- Short Description
- Target Customers
- Problem Solved
- Revenue Model
- Why It Has Potential

Keep the response concise, actionable, and realistic.""")
)

In [ ]:
idea_chain = idea_prompt | llm | parse_and_log_output_chain

In [ ]:
idea_result = idea_chain.invoke("agro")

In [ ]:
idea_result["output"]

In [ ]:
logs